In [1]:
import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
import aplpy

import pandas as pd
import sys
from pathlib import Path
import glob

In [2]:
catalogue_path = '/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/Bubble_Catalogue/infer_catalogue_all.csv' # 提供されたCSVファイル
catalogue_data = pd.read_csv(catalogue_path)

In [3]:
fgn_path_all = glob.glob("/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/FUGIN/12CO/*.fits")
fgn_path_all.sort()

In [4]:
fits_path = "/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/Galaxy_Plane/FUGIN/12CO/FGN_01400+0000_2x2_12CO_v1.00_cube.fits"
integ_fits_path = "/home/cygnus/fujimoto/Cygnus-X_Molecular_Cloud_Analysis/fits/processed_fits/FUGIN/mom0/FGN_01400+0000_2x2_12CO_v1.00_cube.pk_vsmooth_2.0_xysmooth_2.0_thresh_700.0_sigma_3.0_mom0.fits"

integ_hdu = fits.open(integ_fits_path)[0]
hdu = fits.open(fits_path)[0]
wcs = WCS(integ_hdu.header)

raw_d = hdu.data
header = hdu.header

In [6]:
catalogue_data['GLON_center'] = (catalogue_data['ra_min'] + catalogue_data['ra_max']) / 2
catalogue_data['GLAT_center'] = (catalogue_data['dec_min'] + catalogue_data['dec_max']) / 2

In [ ]:
ny, nx = raw_d.shape[1:3]
glon_min, glat_min = wcs.all_pix2world(nx, 0, 0)
glon_max, glat_max = wcs.all_pix2world(0, ny, 0)

# fitsが担当する領域を調整
#  FGN_01100➡️l=10°~12°→10°~11.5°
#  FGN_01200➡️l=11°~13°→11.5°~12.5°
#  FGN_01300➡️l=12°~14°→12.5°~13.5°
# ……

if glon_min > 10:
    glon_min += 0.5
glon_max -= 0.5

print("glon_min: ", glon_min,"\n",
      "glon_max: ", glon_max,"\n",
      "glat_min: ", glat_min,"\n",
      "glat_max: ", glat_max)

catalogue_data_selected = catalogue_data.query(
    f"{glon_min} <= GLON_center and GLON_center <= {glon_max} and {glat_min} <= GLAT_center and GLAT_center <= {glat_max}"
).reset_index()

babble_region_galactic = [] # 銀河座標を格納するリスト
for index, row in catalogue_data_selected.iterrows():
    glon_min = row['ra_min'] # カタログにはなぜかra, decの場所にglon, glatの座標が入っている
    glon_max = row['ra_max']
    glat_min = row['dec_min']
    glat_max = row['dec_max']

    # (l, b) のリストを作成
    babble_region_galactic.append([
        [glon_min, glat_min], # 0: L_min, B_min
        [glon_max, glat_min], # 1: L_max, B_min
        [glon_min, glat_max], # 2: L_min, B_max
        [glon_max, glat_max]  # 3: L_max, B_max
    ])
print(f"babble_region_galactic len: {len(babble_region_galactic)}")

# --- babble_region_pix への変換 (修正) ---
# ループ元を babble_region_galactic に変更
babble_region_pix = []
for i in range(len(babble_region_galactic)):
    babble_region = babble_region_galactic[i] # 銀河座標のバブル領域
    region_list = []
    for j, world_coords_list in enumerate(babble_region):
        world_coords_list = [babble_region[j]] # [L, B] のリスト

        # WCS変換。世界座標 (L, B) をピクセル座標に変換
        # origin=0 でPythonの0-based indexに対応
        region_pix_result = wcs.wcs_world2pix(world_coords_list, 0)

        # 変換結果の形状と要素数をチェックする
        if len(region_pix_result) > 0 and len(region_pix_result[0]) == 2:
            region_list.append(region_pix_result[0]) # 期待通り [x, y] を追加
        else:
            print(f"警告: bubble_num={i}, point_idx={j} のWCS変換結果が不正です: {region_pix_result}. NaNで埋めます。")
            region_list.append([np.nan, np.nan]) # NaNで埋めて、後で除外されるようにする

    babble_region_pix.append(region_list)
print(f"babble_region_pix len: {len(babble_region_pix)}")